In [0]:
%sql
CREATE OR REPLACE TABLE workspace.investment_vision.history_table_january (
    -- Activity_Date DATE,
    -- Process_Date DATE,
    Settle_Date DATE,
    Settle_Month_Year STRING,
    Instrument STRING,
    -- Description STRING,
    Trans_Code STRING,
    Quantity String,
    Price STRING,
    Amount STRING
)
USING DELTA

In [0]:
%sql
select * from investment_vision.raw_statement_activity

In [0]:
%sql
select * from investment_vision.history_table;

In [0]:
%sql
MERGE INTO investment_vision.history_table_january AS target
USING (
    SELECT
        to_date(rsa.date, 'MM/dd/yyyy') AS Settle_Date,
        date_format(to_date(rsa.date, 'MM/dd/yyyy'), 'MMM-yyyy') AS Settle_Month_Year,
        rsa.ticker AS Instrument,
        rsa.transaction_type AS Trans_Code,
        rsa.qty AS Quantity,
        rsa.current_price AS Price,
        rsa.amount AS Amount
    FROM investment_vision.raw_statement_activity rsa
    WHERE to_date(rsa.date, 'MM/dd/yyyy') IS NOT NULL
    UNION ALL
    SELECT
        ri.Settle_Date AS Settle_Date,
        date_format(ri.Settle_Date, 'MMM-yyyy') AS Settle_Month_Year,
        ri.Instrument AS Instrument,
        ri.Trans_Code AS Trans_Code,
        ri.Quantity AS Quantity,
        ri.Price AS Price,
        ri.Amount AS Amount
    FROM investment_vision.history_table ri
    WHERE ri.Settle_Date IS NOT NULL
) AS source
ON 
   target.Quantity = source.Quantity
   AND target.Settle_Date = source.Settle_Date
WHEN NOT MATCHED THEN
    INSERT (
        Instrument, Trans_Code, Quantity, Price, Amount, Settle_Date, Settle_Month_Year
    ) VALUES (
        source.Instrument, source.Trans_Code, source.Quantity, source.Price, source.Amount, source.Settle_Date, source.Settle_Month_Year
    )

In [0]:
%sql
select * from investment_vision.history_table_january